# 02: Structured Outputs

In the last notebook, Claude returned **free text** — paragraphs you could read but couldn't easily process with code. In this notebook, you'll learn to get **structured data** back from Claude — specifically JSON — which you can parse, filter, store, and feed into downstream analysis.

This is where the API becomes a real research tool: Claude reads unstructured text (papers, abstracts, notes) and produces structured data your code can work with.

> **Navigation:** [← Previous: Your First API Call](../08-claude-api/01-your-first-api-call.ipynb) | [Module Index](../README.md) | [Next: Building Research Tools →](../08-claude-api/03-building-research-tools.ipynb)

In [ ]:
from anthropic import Anthropic
import json

client = Anthropic()
print("Ready.")

---

## Quick review: JSON and Python dictionaries

JSON (JavaScript Object Notation) is the standard format for exchanging structured data. If you know Python dictionaries, you already know JSON — they look almost identical.

```json
{
    "gene": "SCN9A",
    "protein": "NaV1.7",
    "role": "nociceptor excitability",
    "pain_disorders": ["erythromelalgia", "PEPD", "CIP"]
}
```

In Python:
- `json.loads(text)` converts a JSON string into a Python dictionary
- `json.dumps(data, indent=2)` converts a dictionary into a formatted JSON string

> **Think Python reference:** Dictionaries (Chapter 11) and string methods. JSON is just a text representation of a dictionary.

---

## Getting JSON from Claude

The trick is simple: **tell Claude to respond in JSON** using the system prompt. Be specific about the exact fields you want.

In [ ]:
message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system=(
        "You are a pain biology expert. Respond ONLY with valid JSON, no other text. "
        "Do not wrap the JSON in markdown code fences."
    ),
    messages=[{
        "role": "user",
        "content": (
            "Provide information about SCN9A in this exact JSON format:\n"
            '{"gene": "...", "protein": "...", "chromosome": "...", '
            '"expression_sites": ["..."], "pain_disorders": ["..."], '
            '"therapeutic_relevance": "..."}'
        )
    }]
)

raw_text = message.content[0].text
print("Raw response:")
print(raw_text)

In [ ]:
# Parse the JSON string into a Python dictionary
data = json.loads(raw_text)

# Now you can access fields programmatically
print(f"Gene: {data['gene']}")
print(f"Protein: {data['protein']}")
print(f"Expression sites: {', '.join(data['expression_sites'])}")
print(f"Number of associated pain disorders: {len(data['pain_disorders'])}")

That's the core pattern: **prompt for JSON** -> **get text back** -> **parse with `json.loads()`** -> **use as a Python dictionary**.

### Structured Output Pipeline

```mermaid
graph LR
    A["Unstructured Input<br>(text, abstract,<br>notes)"] --> B["System Prompt<br>Defines JSON schema<br>+ extraction rules"]
    B --> C["Claude API<br>Extracts structured<br>data from text"]
    C --> D["Raw JSON String<br>'{\"gene\": \"SCN9A\", ...}'"]
    D --> E["json.loads()<br>Parse to Python"]
    E --> F["Python Dict<br>{'gene': 'SCN9A', ...}"]
    F --> G["pd.DataFrame()<br>Tabular analysis"]

    style A fill:#E24A33,color:#fff
    style B fill:#FDB863,color:#000
    style C fill:#4878CF,color:#fff
    style D fill:#eee,color:#000
    style E fill:#eee,color:#000
    style F fill:#55A868,color:#fff
    style G fill:#55A868,color:#fff
```

This is the core pattern for turning unstructured text into structured data. The system prompt is your schema definition -- it tells Claude exactly what fields to extract and what format to use. The parsed dictionary can then be combined with other results into a pandas DataFrame for analysis.

---

## Robust JSON parsing

Sometimes Claude wraps JSON in markdown code fences (` ```json ... ``` `) even when you ask it not to. Let's write a helper that handles this reliably.

In [ ]:
def parse_json_response(text):
    """Parse JSON from Claude's response, handling markdown code fences."""
    # Strip markdown code fences if present
    cleaned = text.strip()
    if cleaned.startswith("```"):
        # Remove first line (```json) and last line (```)
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:-1])
    
    return json.loads(cleaned)


# Test with a response that has code fences
test_with_fences = '```json\n{"gene": "TRPV1", "role": "heat sensing"}\n```'
test_without = '{"gene": "TRPV1", "role": "heat sensing"}'

print(parse_json_response(test_with_fences))
print(parse_json_response(test_without))

---

## System prompts for structured output

The system prompt is where you define the output schema. Here are patterns that work well:

### Pattern 1: Give an example in the system prompt

In [ ]:
EXTRACTION_SYSTEM = """You are a research data extraction assistant. 
Given information about an ion channel, extract structured data.

Respond ONLY with valid JSON in this exact format (no other text, no code fences):
{
    "channel_name": "common name (e.g., NaV1.7)",
    "gene": "gene symbol",
    "channel_type": "sodium|potassium|calcium|chloride|TRP|other",
    "primary_expression": ["list of tissues/cell types"],
    "pain_relevant": true or false,
    "mechanism_in_pain": "brief description or null if not pain-relevant",
    "druggable": true or false
}"""

message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    system=EXTRACTION_SYSTEM,
    messages=[{"role": "user", "content": "Tell me about KCNQ2 (Kv7.2)"}]
)

channel_data = parse_json_response(message.content[0].text)
print(json.dumps(channel_data, indent=2))

### Pattern 2: Use prefilling to guarantee JSON

A powerful technique: put the opening `{` as an **assistant message**, so Claude is forced to continue the JSON object.

In [ ]:
message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    system="Extract ion channel information as JSON with fields: channel_name, gene, channel_type, pain_relevant (bool), key_function.",
    messages=[
        {"role": "user", "content": "Extract info about TRPV1."},
        {"role": "assistant", "content": "{"}  # prefill forces JSON output
    ]
)

# Since we prefilled with "{", we need to add it back
raw = "{" + message.content[0].text
data = json.loads(raw)
print(json.dumps(data, indent=2))

The prefilling technique is useful when you absolutely need clean JSON with no preamble. For most cases, a clear system prompt is enough.

---

## Building a pipeline: text in, structured data out

Now let's combine everything into a real function. Imagine you have a DRG RNA-seq experiment and you want to quickly annotate a list of differentially expressed genes.

In [ ]:
def annotate_gene(gene_name):
    """Use Claude to annotate a gene with pain biology context."""
    system = """You are a pain biology annotation tool. Given a gene name, return JSON with:
{
    "gene": "gene symbol",
    "protein": "protein name",
    "category": "ion_channel|receptor|kinase|transcription_factor|neuropeptide|cytokine|other",
    "pain_relevant": true/false,
    "pain_role": "brief description or null",
    "expression_in_drg": "high|moderate|low|absent|unknown"
}
Respond ONLY with valid JSON, no other text, no code fences."""
    
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        system=system,
        messages=[{"role": "user", "content": f"Annotate: {gene_name}"}]
    )
    
    return parse_json_response(message.content[0].text)


# Test with a single gene
result = annotate_gene("CALCA")
print(json.dumps(result, indent=2))

In [ ]:
# Now annotate several genes from a hypothetical DRG RNA-seq DE list
de_genes = ["SCN9A", "TRPV1", "CALCA", "IL6", "NGF", "NTRK1"]

annotations = []
for gene in de_genes:
    print(f"Annotating {gene}...", end=" ")
    ann = annotate_gene(gene)
    annotations.append(ann)
    print(f"-> {ann['category']}, pain_relevant={ann['pain_relevant']}")

print(f"\nAnnotated {len(annotations)} genes.")

In [ ]:
# Filter: which of these are pain-relevant ion channels or receptors?
pain_hits = [
    a for a in annotations 
    if a["pain_relevant"] and a["category"] in ("ion_channel", "receptor")
]

print("Pain-relevant ion channels/receptors in DE gene list:")
for hit in pain_hits:
    print(f"  {hit['gene']} ({hit['protein']}) — {hit['pain_role']}")

This is the basic pipeline: **list of inputs** -> **Claude annotates each** -> **filter/analyze the structured results**. You'll build more sophisticated versions of this in the next notebook.

---

## Handling parsing errors gracefully

Sometimes Claude's JSON isn't perfect. Always wrap parsing in a try/except so one bad response doesn't crash your whole pipeline.

In [ ]:
def safe_annotate_gene(gene_name):
    """Annotate a gene, returning None if parsing fails."""
    try:
        return annotate_gene(gene_name)
    except json.JSONDecodeError as e:
        print(f"  WARNING: Failed to parse JSON for {gene_name}: {e}")
        return None
    except Exception as e:
        print(f"  WARNING: API error for {gene_name}: {e}")
        return None


# This version won't crash if one gene fails
result = safe_annotate_gene("SCN10A")
if result:
    print(json.dumps(result, indent=2))
else:
    print("Failed — would skip this gene and continue.")

---

## Putting structured data into a pandas DataFrame

Since you'll often want to analyze results in a table, here's how to go from a list of JSON responses to a DataFrame.

In [ ]:
import pandas as pd

# We already have the annotations list from above
df = pd.DataFrame(annotations)
df

In [ ]:
# Now you can use standard pandas operations
print("Pain-relevant genes:")
print(df[df["pain_relevant"] == True][["gene", "protein", "category", "pain_role"]])

print(f"\nCategory counts:")
print(df["category"].value_counts())

---

## Exercise: Paper abstract extractor

Build a function called `extract_paper_info` that takes a paper **title** and **abstract** and returns a dictionary with these fields:

- `target` — the molecular target studied (gene or protein name)
- `mechanism` — the biological mechanism investigated
- `model_organism` — the species or model system used
- `pain_type` — type of pain studied (inflammatory, neuropathic, acute, etc.)
- `key_finding` — one-sentence summary of the main result
- `techniques` — list of experimental techniques used

Test it with the example abstracts provided below.

In [ ]:
# Your code here

def extract_paper_info(title, abstract):
    """Extract structured information from a paper title and abstract."""
    system = """You are a pain biology literature extraction tool. Given a paper title and
abstract, extract structured data. Respond ONLY with valid JSON (no other text, no code fences):
{
    "target": "molecular target (gene or protein)",
    "mechanism": "biological mechanism investigated",
    "model_organism": "species or model system",
    "pain_type": "inflammatory|neuropathic|acute|cancer|visceral|other",
    "key_finding": "one-sentence summary of main result",
    "techniques": ["list", "of", "techniques"]
}
If a field cannot be determined from the abstract, use null."""
    
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system=system,
        messages=[{
            "role": "user",
            "content": f"Title: {title}\n\nAbstract: {abstract}"
        }]
    )
    
    return parse_json_response(message.content[0].text)

In [ ]:
# Test abstract 1
title1 = "NaV1.7 gain-of-function mutations in idiopathic small fiber neuropathy"
abstract1 = """Small fiber neuropathy (SFN) often remains idiopathic despite extensive 
evaluation. We hypothesized that gain-of-function mutations in sodium channel NaV1.7 
might be present in patients with idiopathic SFN. We sequenced SCN9A in 28 patients 
with biopsy-confirmed idiopathic SFN and identified mutations in 8 patients (28.6%). 
Functional analysis using voltage-clamp recordings in DRG neurons transfected with 
mutant channels showed gain-of-function changes including hyperpolarized activation 
and enhanced ramp currents. These mutations rendered DRG neurons hyperexcitable as 
demonstrated by current-clamp recordings showing reduced current threshold and 
increased firing frequency. Our results suggest that gain-of-function mutations in 
NaV1.7 are a common cause of SFN and may provide targets for pain treatment."""

result1 = extract_paper_info(title1, abstract1)
print(json.dumps(result1, indent=2))

In [ ]:
# Test abstract 2
title2 = "CGRP receptor antagonism in migraine treatment"
abstract2 = """Calcitonin gene-related peptide (CGRP) plays a critical role in migraine 
pathophysiology. We conducted a randomized, double-blind, placebo-controlled trial of 
a monoclonal antibody targeting the CGRP receptor in 955 patients with episodic migraine. 
Patients receiving the active treatment showed a mean reduction of 4.2 migraine days per 
month compared to 2.1 for placebo (P<0.001). Adverse events were similar between groups. 
Calcium imaging of trigeminal neurons in a mouse model confirmed reduced neuronal 
activation following CGRP receptor blockade."""

result2 = extract_paper_info(title2, abstract2)
print(json.dumps(result2, indent=2))

In [ ]:
# Combine results into a DataFrame for comparison
papers_df = pd.DataFrame([result1, result2])
papers_df

---

## Exercise: Try it with your own abstracts

Copy-paste a real abstract from a paper relevant to your work and run it through `extract_paper_info`. How well does it capture the key information?

In [ ]:
# Paste your own abstract here and test
# my_title = "..."
# my_abstract = "..."
# result = extract_paper_info(my_title, my_abstract)
# print(json.dumps(result, indent=2))

---

## What you just learned

- **JSON from Claude** — use system prompts to request specific JSON schemas
- **Parsing** — `json.loads()` converts text to Python dictionaries, with `parse_json_response()` handling markdown fences
- **Prefilling** — put `{` as an assistant message to force clean JSON
- **Pipelines** — input -> Claude -> structured data -> analysis
- **Error handling** — `try/except` around parsing so one failure doesn't crash a batch
- **DataFrames** — `pd.DataFrame(list_of_dicts)` for instant tabular analysis

Next up: **building research tools** — batch processing, multi-turn conversations, and combining Claude with pandas for real data workflows.

> **Navigation:** [← Previous: Your First API Call](../08-claude-api/01-your-first-api-call.ipynb) | [Module Index](../README.md) | [Next: Building Research Tools →](../08-claude-api/03-building-research-tools.ipynb)

---

## Edit Log

- 2026-03-25: Created notebook with initial content
- 2026-03-25: Added cross-module navigation links